# gufe-viz in a notebook

`gufe_viz.view(obj)` puts **the page the CLI writes** into a cell. Same bundle,
same `<gufe-view>`, same drawing code - the only thing that changes between a
notebook, `gufe-viz obj.json -o page.html` and a future `openfe view` is how the
payload reaches the element.

This notebook is the test bench for that: **every payload type** and **every way
of delivering one**, in one file.

```
pixi run notebook     # JupyterLab, on this file
pixi run marimo       # the same notebook, converted, in marimo
```

> **This is the notebook you run.** It is committed with no outputs and should
> stay that way - each view's output is a whole page in an iframe, which adds a
> quarter of a megabyte per cell and renders as a blank on GitHub regardless.
> Clear outputs before committing.
>
> Its counterpart, [`gufe-viz-gallery.ipynb`](./gufe-viz-gallery.ipynb), is the
> one to *look* at: the same views as screenshots, so they are visible on GitHub
> to someone who is not going to install anything. See
> [`README.md`](./README.md) for which to reach for when.

**What a cell gets.** Two layers come out of one `view()` call:

| layer | mimetype | needs | gives |
| --- | --- | --- | --- |
| static | `text/html` - the page in an `<iframe srcdoc>` | nothing | a saved notebook that still draws with no kernel |
| live | a widget view - shell page + payload as widget state | `anywidget` | `w.payload = other` redraws in place |

Your frontend picks. With a live kernel you get the widget; `nbconvert`, nbviewer
and a mailed `.ipynb` fall back to the page. If `anywidget` is not installed you
get the static layer alone and everything below still draws - except the one
cell that updates in place, which is what the widget is for.

## Setup

The golden payloads in `examples/` - the same files pytest, vitest and the
drag-and-drop dev page read. No gufe needed for this half.

In [ ]:
import json
from pathlib import Path

import gufe_viz

root = Path.cwd()
while not (root / "examples").is_dir() and root != root.parent:
    root = root.parent
EXAMPLES = root / "examples"

payloads = {p.stem: json.loads(p.read_text()) for p in sorted(EXAMPLES.glob("*.json"))}

try:
    import anywidget  # noqa: F401

    live = f"yes (anywidget {anywidget.__version__})"
except ImportError:
    live = "no - static pages only. `pip install gufe-viz[notebook]`"

print(f"gufe-viz {gufe_viz.__version__}")
print(f"live widgets: {live}")
print()
for name, payload in payloads.items():
    print(f"  {name:24} {payload['type']}")

---

# Every payload type

One cell per example, so you can re-run just the one you are working on.

Three payload types are drawn today - `SmallMoleculeComponentViz`,
`ProteinComponentViz`, `LigandNetworkViz`. Everything else, **including the
protein subclasses**, reaches the dispatcher and gets the
**"no visualization for X yet"** panel, which is not a failure: it is the
graceful-degradation rule doing its job, and it is worth looking at as often as
the pictures are.

### `small_molecule.json`

2D RDKit depiction beside the 3D conformer, with SMILES, charge and atom counts.

In [ ]:
gufe_viz.view(payloads["small_molecule"])

### `small_molecule_charged.json`

The same view with a non-zero formal charge.

In [ ]:
gufe_viz.view(payloads["small_molecule_charged"])

### `protein.json`

3Dmol with representation and colour-scheme switchers; waters hidden by default.

In [ ]:
gufe_viz.view(payloads["protein"])

### `protein_fragment.json`

A small protein, so the 3D view loads fast while iterating.

In [ ]:
gufe_viz.view(payloads["protein_fragment"])

### `protein_membrane.json`

**Not drawn.** Python's MRO walk gives a membrane system the protein *builder*, but the payload says `ProteinMembraneComponentViz` and the browser's dispatch table has no entry for that - inheritance on one side of the contract is not inheritance on the other.

In [ ]:
gufe_viz.view(payloads["protein_membrane"])

### `solvated_pdb.json`

**Not drawn**, for the same reason as `protein_membrane`.

In [ ]:
gufe_viz.view(payloads["solvated_pdb"])

### `ligand_network.json`

Radial graph of the ligands, atom mapping on the right for the selected edge.

In [ ]:
gufe_viz.view(payloads["ligand_network"])

### `ligand_network_named.json`

The same network with the ligands named, so labels replace gufe keys.

In [ ]:
gufe_viz.view(payloads["ligand_network_named"])

### `chemical_system.json`

Not drawn yet - the dispatcher's panel.

In [ ]:
gufe_viz.view(payloads["chemical_system"])

### `ligand_atom_mapping.json`

Not drawn yet - the standalone mapping viewer is Phase 4.

In [ ]:
gufe_viz.view(payloads["ligand_atom_mapping"])

### `solvent.json`

Not drawn yet - a solvent component is a specification, so its view is a settings card.

In [ ]:
gufe_viz.view(payloads["solvent"])

---

# From live gufe objects

The same call, handed real objects rather than payload JSON. `view()` serializes
through `gufe_viz.payload_for`, so this is the path a user actually takes.

In [ ]:
import warnings

import gufe

warnings.filterwarnings("ignore", message=".*hydrogen atoms.*")

data = Path(gufe.__file__).parent / "tests" / "data"

net = gufe.LigandNetwork.from_graphml((data / "ligand_network.graphml").read_text())
edge = sorted(net.edges, key=lambda e: str(e.key))[0]
ligand_A, ligand_B = edge.componentA, edge.componentB

protein = gufe.ProteinComponent.from_pdb_file(str(data / "181l.pdb"), name="T4 lysozyme")
solvent = gufe.SolventComponent()

stateA = gufe.ChemicalSystem({"ligand": ligand_A, "protein": protein, "solvent": solvent}, name="complex A")
stateB = gufe.ChemicalSystem({"ligand": ligand_B, "protein": protein, "solvent": solvent}, name="complex B")

# A Transformation needs a Protocol, and gufe's own dummy is the one that needs
# no simulation engine installed. It lives under gufe's tests, so treat it as
# optional rather than as something to fail the notebook over.
try:
    from gufe.tests.test_protocol import DummyProtocol

    protocol = DummyProtocol(settings=DummyProtocol.default_settings())
    transformation = gufe.Transformation(stateA=stateA, stateB=stateB, protocol=protocol, mapping=edge, name="A to B")
    alchemical_net = gufe.AlchemicalNetwork(edges=[transformation], name="demo campaign")
except ImportError as e:
    transformation = alchemical_net = None
    print(f"no DummyProtocol ({e}) - the transformation cells will be skipped")

label = lambda component: component.name or str(component.key)  # noqa: E731
print(f"ligands   {label(ligand_A)} -> {label(ligand_B)}")
print(f"network   {len(net.nodes)} ligands, {len(net.edges)} edges")

### `SmallMoleculeComponent`

In [ ]:
gufe_viz.view(ligand_A)

### `ProteinComponent`, with a size override

In [ ]:
gufe_viz.view(protein, height="500px")

### `LigandNetwork`

In [ ]:
gufe_viz.view(net)

### `ChemicalSystem`, `LigandAtomMapping`, `SolventComponent`

All three reach the panel today.

In [ ]:
gufe_viz.view(stateA)

### `Transformation` and `AlchemicalNetwork`

In [ ]:
gufe_viz.view(transformation) if transformation is not None else "no DummyProtocol - skipped"

In [ ]:
gufe_viz.view(alchemical_net) if alchemical_net is not None else "no DummyProtocol - skipped"

---

# Every way of delivering one

## 1. Static only

`live=False` skips anywidget even when it is installed. This is exactly what a
reader with no kernel sees, so it is the honest preview of an export.

In [ ]:
gufe_viz.view(ligand_A, live=False, height="420px")

## 2. Live, and updated in place

Run the next cell, then the one after it, and **watch the output above change**
without re-running it. That is the whole point of the live layer: the element's
update beat, driven from Python.

In [ ]:
w = gufe_viz.view(ligand_A, height="420px")
w

In [ ]:
w.payload = ligand_B  # scroll up: the cell above is now showing ligand B
print("showing:", w.payload.get("name") or w.payload["type"])

## 3. Live, without the exported copy

`static=False` drops the `text/html` layer. The cell halves what it costs to
display, and an exported notebook has a hole where it was. See the sizes below
for why the knob exists.

In [ ]:
gufe_viz.view(net, static=False, height="420px")

## 4. The file the CLI writes

`to_html` returns a string and writes nothing; the CLI is one answer to where it
goes, and so is this cell.

In [ ]:
import tempfile

scratch = Path(tempfile.gettempdir())

out = scratch / "protein.html"
out.write_text(gufe_viz.to_html(payloads["protein"]), encoding="utf-8")
print(f"wrote {out} ({out.stat().st_size:,} bytes) - open it in a browser")

## 5. ...via the CLI itself

The same page, from the shell. `gufe-viz` is a development convenience rather
than the OpenFE CLI integration, but it is the same `to_html` underneath.

In [ ]:
source = EXAMPLES / "small_molecule.json"
target = scratch / "small_molecule.html"
!gufe-viz "{source}" -o "{target}"

## 6. What a view costs

Every displayed view sends the bundle to the browser: once in the page when
`static`, once in the widget's shell when live, and both when both. On
JupyterLab those messages share the kernel's iopub channel, which the server
rate-limits by default - so a notebook that creates many views in one burst can
have messages **dropped** rather than delivered slowly, and cells come up blank.
`static=False` and `live=False` are the two knobs.

In [ ]:
page = gufe_viz.to_html(payloads["protein"])
shell = gufe_viz.shell_html()

print(f"bundle                {len(gufe_viz.bundle_source()):>10,} bytes")
print(f"payload (protein)     {len(json.dumps(payloads['protein'])):>10,} bytes")
print(f"static page           {len(page):>10,} bytes  <- per view, in the .ipynb")
print(f"widget shell          {len(shell):>10,} bytes  <- per view, over the comm")
print(f"both (the default)    {len(page) + len(shell):>10,} bytes")

## 7. When it cannot draw

Three different situations, three different answers. Only the first is an
exception, because only the first is a mistake at the call site.

In [ ]:
# (a) something gufe-viz has no view for at all: a mistake at the call site, so
#     it raises rather than drawing a panel about it
try:
    gufe_viz.view(object())
except TypeError as e:
    print("TypeError:", e)

In [ ]:
# (b) a payload whose type has no view yet: the dispatcher's panel, no exception
gufe_viz.view({"type": "SolventComponentViz", "name": "water"}, height="180px")

In [ ]:
# (c) a type nobody ever declared: still a panel, and it says which of the two
#     problems it is
gufe_viz.view({"type": "NotAThing"}, height="180px")

---

## Notes

**Why an iframe, and not the cell's own DOM.** Two properties of the bundle, not
caution: the `<gufe-*>` elements build light DOM, so a notebook's output-area CSS
would reach inside every view; and the engine loaders append a `<script>` to
`document.head` and read `window.$3Dmol` / `window.RDKit`, which in a notebook
page are the globals py3Dmol and nglview are already using, possibly at another
version. The iframe settles both for nothing.

**Engines come from a CDN.** RDKit and 3Dmol load on demand, from a view that
needs them - so a cell offline shows the page, the layout and the metadata, but
no depiction and no 3D viewer. `to_html(..., engines="bundled")` will close that
half once it exists.

**marimo.** `pixi run marimo` converts this file and opens it. The static layer
is plain HTML and needs nothing; the live layer goes through marimo's own
anywidget support.